<a href="https://colab.research.google.com/github/Anast2007/UNIT_TEST-1---DL/blob/main/Household%20Power.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("uciml/electric-power-consumption-data-set")

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# List files in the downloaded dataset path
print("Files in dataset path:", os.listdir(path))


In [ ]:
# Find the text/csv file and load it into a pandas DataFrame
# The UC Irvine Household Electric Power Consumption dataset is typically a ';' separated file
file_name = [f for f in os.listdir(path) if f.endswith('.txt') or f.endswith('.csv')][0]
full_file_path = os.path.join(path, file_name)

print(f"Loading dataset from {full_file_path}...")
# Reading with sep=';' and low_memory=False as this dataset is relatively large (~2M rows)
df = pd.read_csv(full_file_path, sep=';', low_memory=False)

# Display basic information
print(df.info())
display(df.head())


In [ ]:
# In this dataset, missing values are known to be marked as '?'
# We will replace '?' with NaN and convert the numerical columns to float
numeric_cols = [
    'Global_active_power', 'Global_reactive_power', 'Voltage',
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace('?', 'nan', regex=False), errors='coerce')

# Impute missing values (using forward fill as this is a sequential time-series dataset)
print("Missing values before imputation:\n", df[numeric_cols].isnull().sum())
df[numeric_cols] = df[numeric_cols].ffill()
print("\nMissing values after imputation:\n", df[numeric_cols].isnull().sum())


In [ ]:
# Normalize the numerical features using MinMaxScaler
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[numeric_cols])

# Create a new DataFrame with the normalized features
df_normalized = df.copy()
df_normalized[numeric_cols] = scaled_features

print("Normalized features summary:")
display(df_normalized[numeric_cols].describe())


### Line-by-Line Explanation of Sequence Generation

To build sequences for training models (like LSTMs or RNNs) from our continuous time-series data, we use a sliding window approach:

1. **`def create_sequences(data, seq_length):`**  
   Defines a function that accepts our numerical dataset (`data`) and the length of history we want to look back at (`seq_length`).

2. **`X, y = [], []`**  
   Initializes empty lists to store the input features sequences ($X$) and target steps ($y$).

3. **`for i in range(len(data) - seq_length):`**  
   Loops through the records. We stop `seq_length` rows before the end of the dataset so that each input sequence has a valid, succeeding target index to predict.

4. **`X.append(data[i : i + seq_length])`**  
   Extracts a block of consecutive rows from index `i` to `i + seq_length` (exclusive) and adds it as one sequence step inside $X$.

5. **`y.append(data[i + seq_length, 0])`**  
   Identifies the target to predict. In this example, we select index `0` (`Global_active_power`) at the precise time step immediately following the input window (`i + seq_length`) and add it to $y$.

6. **`return np.array(X), np.array(y)`**  
   Converts the lists to optimized numpy arrays suitable for model ingestion.

In [ ]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        # Slice historical window of features
        X.append(data[i : i + seq_length])
        # Define target as the next step's Global_active_power (index 0)
        y.append(data[i + seq_length, 0])
    return np.array(X), np.array(y)

# Convert normalized dataframe's features to a numpy array
data_array = df_normalized[numeric_cols].values

# Define the sequence length (using the last 60 minutes to predict the next minute)
SEQUENCE_LENGTH = 60

# Generate sequences using the first 10,000 samples for demonstration purposes
X_samples, y_samples = create_sequences(data_array[:10000], SEQUENCE_LENGTH)

print(f"Generated features shape (X): {X_samples.shape} -> (samples, time_steps, features)")
print(f"Generated target shape (y):   {y_samples.shape} -> (samples,)")


### Selection and Justification of the Recurrent Model

For forecasting continuous household electric power consumption, we select a **Long Short-Term Memory (LSTM)** network over a standard Recurrent Neural Network (RNN).

#### Why LSTM?
1. **Long-Term Dependencies**: Standard RNNs struggle with vanishing gradients when training on long sequences (like our window of 60 minutes). LSTMs use memory cells and gating mechanisms (input, forget, and output gates) to retain historical patterns over long time horizons.
2. **Temporal Modeling**: Power consumption exhibits complex cyclic patterns (daily routines, appliance usage cycles). LSTMs excel at learning when to keep or discard temporal signals across sequence steps.
3. **Multivariate Support**: It can effortlessly process all 7 parallel input features to predict our single target variable (`Global_active_power`).

In [ ]:
# @title
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from sklearn.model_selection import train_test_split

# Generate sequences using a larger subset (first 150,000 samples) to ensure stable training
X_full, y_full = create_sequences(data_array[:150000], SEQUENCE_LENGTH)

# Split sequence data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, shuffle=True
)

# Define the updated LSTM Model architecture utilizing Input layer to resolve the warning
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

# Compile the model with Adam optimizer and Mean Squared Error loss
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_val, y_val),
    verbose=1
)

In [ ]:
X_full_25k, y_full_25k = create_sequences(data_array[:25000], SEQUENCE_LENGTH)

X_train, X_val, y_train, y_val = train_test_split(
    X_full_25k, y_full_25k, test_size=0.2, random_state=42, shuffle=True
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")

model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_val, y_val),
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Set up a side-by-side plot for Loss and MAE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot training & validation loss (MSE)
ax1.plot(history.history['loss'], label='Train Loss (MSE)')
ax1.plot(history.history['val_loss'], label='Val Loss (MSE)')
ax1.set_title('Model Loss (MSE) over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# Plot training & validation MAE
ax2.plot(history.history['mae'], label='Train MAE')
ax2.plot(history.history['val_mae'], label='Val MAE')
ax2.set_title('Model Mean Absolute Error (MAE) over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predict on the validation set
y_pred = model.predict(X_val)

# Calculate metrics (in the scaled domain)
val_mae = mean_absolute_error(y_val, y_pred)
val_mse = mean_squared_error(y_val, y_pred)
val_rmse = np.sqrt(val_mse)

print(f"Validation Mean Absolute Error (MAE): {val_mae:.5f}")
print(f"Validation Root Mean Squared Error (RMSE): {val_rmse:.5f}")

In [ ]:
import matplotlib.pyplot as plt

# We will plot the first 200 validation samples to clearly visualize the sequence predictions
plt.figure(figsize=(15, 6))
plt.plot(y_val[:200], label='Actual Consumption', color='blue', alpha=0.7, linewidth=1.5)
plt.plot(y_pred[:200], label='Predicted Consumption', color='red', linestyle='--', alpha=0.8, linewidth=1.5)
plt.title('Actual vs Predicted Global Active Power (First 200 Unseen Validation Samples)')
plt.xlabel('Sample Index')
plt.ylabel('Normalized Consumption Value')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Calculate Mean Absolute Percentage Error (MAPE) and convert it to a percentage accuracy
# To avoid division by zero on scaled values, we add a tiny epsilon
epsilon = 1e-5
mape = np.mean(np.abs((y_val - y_pred.flatten()) / (y_val + epsilon))) * 100
mape_accuracy = 100 - mape

# Calculate Directional Accuracy: Did we correctly predict if consumption goes up or down?
actual_direction = np.diff(y_val) > 0
pred_direction = np.diff(y_pred.flatten()) > 0
directional_accuracy = np.mean(actual_direction == pred_direction) * 100

print(f"Regression Trend-Based Accuracy (100 - MAPE): {mape_accuracy:.2f}%")
print(f"Directional Accuracy (Up/Down Prediction): {directional_accuracy:.2f}%")